## 00. Visualization

In [4]:
import pandas as pd
import dtale
from pathlib import Path

# Function to show df in D-Tale
def show_df_in_dtale(df):
    instance = dtale.show(df)
    instance.kill()
    d = dtale.show(df, enable_custom_filters=True)
    print(f"D-Tale running on: {d._main_url}")


def calculate_target_correlations(
    df: pd.DataFrame,
    target_col: str,
    method: str = "pearson",
    drop_target: bool = True,
) -> pd.DataFrame:
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found in dataset.")

    numeric_df = df.select_dtypes(include="number").copy()

    if target_col not in numeric_df.columns:
        raise ValueError(
            f"Target column '{target_col}' is not numeric or is not available in numeric columns."
        )

    correlations = (
        numeric_df.corr(method=method)[target_col]
        .dropna()
        .rename("correlation")
        .to_frame()
        .reset_index(names="feature")
    )

    if drop_target:
        correlations = correlations.loc[correlations["feature"] != target_col].copy()

    correlations["abs_correlation"] = correlations["correlation"].abs()

    return correlations.sort_values(
        by="abs_correlation",
        ascending=False,
    ).reset_index(drop=True)

In [10]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

#### **Processed for training**

In [ ]:
df = pd.read_csv("../../data/Lecta/processed_for_training/lecta_processed_for_training_month_h6_t_plus_1.csv", sep=";")
df

,code,warehouse,month,value,momentum_ratio_1,momentum_ratio_4,year,week_of_year,month_of_year,quarter_of_year,...,rolling_slope_3_scaled,rolling_slope_6_scaled,rolling_slope_12_scaled,last_year_mean_3_scaled,last_year_median_3_scaled,spike_prior_median_6_scaled,spike_prior_mad_6_scaled,sample_weight,target_t_plus_1,target_scaled_t_plus_1
0,40000892,1201,2023-12-01,3686.4,NaN,NaN,2023,48,12,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.198519,6912.0,1.875000
1,40000892,1201,2024-01-01,6912.0,1.875000,NaN,2024,1,1,1,...,6.086957e-01,0.608696,0.608696,NaN,NaN,NaN,NaN,0.210549,1843.2,0.347826
2,40000892,1201,2024-02-01,1843.2,0.266667,NaN,2024,5,2,1,...,-2.222222e-01,-0.222222,-0.222222,NaN,NaN,NaN,NaN,0.223307,2304.0,0.555556
3,40000892,1201,2024-03-01,2304.0,1.250000,NaN,2024,9,3,1,...,-6.250000e-01,-0.250000,-0.250000,NaN,NaN,NaN,NaN,0.235941,3686.4,1.000000
4,40000892,1201,2024-04-01,3686.4,1.600000,1.000000,2024,14,4,2,...,2.500000e-01,-0.125000,-0.125000,NaN,NaN,0.812500,0.250000,0.250237,2304.0,0.625000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4369,60009778,1204,2025-11-01,115.2,1.333333,0.421053,2025,44,11,4,...,-5.806452e-01,-0.092166,0.080758,0.602151,0.258065,0.903226,0.516129,0.750844,43.2,0.387097
4370,60009778,1204,2025-12-01,43.2,0.375000,1.000000,2025,49,12,4,...,-1.894737e-01,-0.274286,0.017225,0.800000,0.884211,1.010526,0.442105,0.794831,115.2,1.010526
4371,60009778,1204,2026-01-01,115.2,2.666667,0.470588,2026,1,1,1,...,-2.388938e-16,-0.053571,0.012238,0.458333,0.375000,0.875000,0.500000,0.842994,201.6,1.750000
4372,60009778,1204,2026-02-01,201.6,1.750000,2.333333,2026,5,2,1,...,6.168224e-01,-0.044860,0.011372,0.560748,0.560748,0.785047,0.280374,0.894074,201.6,1.570093


In [11]:
correlations_scaled = calculate_target_correlations(
    df=df,
    target_col="target_scaled_t_plus_1",
)

print(correlations_scaled)

                         feature  correlation  abs_correlation
0         rolling_mean_12_scaled    -0.264008         0.264008
1             ewm_mean_12_scaled     0.233177         0.233177
2              ewm_mean_6_scaled     0.150268         0.150268
3      last_year_median_3_scaled     0.125555         0.125555
4                nonzero_rate_12    -0.124070         0.124070
5               log_scale_factor    -0.116842         0.116842
6                 nonzero_rate_6    -0.113928         0.113928
7          time_since_last_spike     0.111971         0.111971
8          rolling_mean_6_scaled    -0.110033         0.110033
9                 nonzero_rate_3    -0.104661         0.104661
10      rolling_median_12_scaled    -0.088447         0.088447
11              is_nonzero_lag_0    -0.086103         0.086103
12   spike_prior_median_6_scaled    -0.081962         0.081962
13       rolling_median_6_scaled    -0.081335         0.081335
14         rolling_mean_4_scaled    -0.080701         0

In [12]:
correlations_scaled.to_csv("correlations_target_scaled_t_plus_1.csv", sep=";", index=False)

In [13]:
correlations_raw = calculate_target_correlations(
    df=df,
    target_col="target_t_plus_1",
)


print(correlations_raw.head(20))

                 feature  correlation  abs_correlation
0      rolling_median_12     0.732387         0.732387
1       rolling_median_6     0.731217         0.731217
2   spike_prior_median_6     0.728538         0.728538
3       rolling_median_4     0.724625         0.724625
4            ewm_mean_12     0.722994         0.722994
5        rolling_mean_12     0.719211         0.719211
6           scale_factor     0.719211         0.719211
7          rolling_min_6     0.711701         0.711701
8             ewm_mean_6     0.710386         0.710386
9         rolling_mean_6     0.709426         0.709426
10         rolling_min_4     0.708148         0.708148
11        rolling_min_12     0.703625         0.703625
12      rolling_median_3     0.699913         0.699913
13        rolling_mean_4     0.697384         0.697384
14         rolling_min_3     0.692697         0.692697
15            ewm_mean_3     0.680152         0.680152
16        rolling_mean_3     0.679867         0.679867
17    last

In [14]:
correlations_raw.to_csv("correlations_target_t_plus_1.csv", sep=";", index=False)
